In [ ]:
# https://github.com/minillinim/ellipsoid
def ellipsoid_plot(center, radii, rotation, ax, plot_axes=False, cage_color='b', cage_alpha=0.2):
    """Plot an ellipsoid"""
        
    u = np.linspace(0.0, 2.0 * np.pi, 100)
    v = np.linspace(0.0, np.pi, 100)
    
    # cartesian coordinates that correspond to the spherical angles:
    x = radii[0] * np.outer(np.cos(u), np.sin(v))
    y = radii[1] * np.outer(np.sin(u), np.sin(v))
    z = radii[2] * np.outer(np.ones_like(u), np.cos(v))
    # rotate accordingly
    for i in range(len(x)):
        for j in range(len(x)):
            [x[i, j], y[i, j], z[i, j]] = np.dot([x[i, j], y[i, j], z[i, j]], rotation) + center

    if plot_axes:
        # make some purdy axes
        axes = np.array([[radii[0],0.0,0.0],
                         [0.0,radii[1],0.0],
                         [0.0,0.0,radii[2]]])
        # rotate accordingly
        for i in range(len(axes)):
            axes[i] = np.dot(axes[i], rotation)

        # plot axes
        for p in axes:
            X3 = np.linspace(-p[0], p[0], 100) + center[0]
            Y3 = np.linspace(-p[1], p[1], 100) + center[1]
            Z3 = np.linspace(-p[2], p[2], 100) + center[2]
            ax.plot(X3, Y3, Z3, color=cage_color)

    # plot ellipsoid
    ax.plot_wireframe(x, y, z,  rstride=4, cstride=4, color=cage_color, alpha=cage_alpha)

In [ ]:
# http://www.mathworks.com/matlabcentral/fileexchange/24693-ellipsoid-fit
# for arbitrary axes
def ellipsoid_fit(X):
    x = X[:, 0]; y = X[:, 1]; z = X[:, 2]
    D = np.array([x*x + y*y - 2*z*z,
                  x*x + z*z - 2*y*y,
                  2*x * y,
                  2*x * z,
                  2*y * z,
                  2*x,
                  2*y,
                  2*z,
                  1 - 0*x])
    d2 = np.array(x*x + y*y + z*z).T # rhs for LLSQ
    u = np.linalg.solve(D.dot(D.T), D.dot(d2))
    
    a = np.array([u[0] + 1*u[1] - 1])
    b = np.array([u[0] - 2*u[1] - 1])
    c = np.array([u[1] - 2*u[0] - 1])
    
    v = np.concatenate([a, b, c, u[2:]], axis=0).flatten()
    
    A = np.array([[v[0], v[3], v[4], v[6]],
                  [v[3], v[1], v[5], v[7]],
                  [v[4], v[5], v[2], v[8]],
                  [v[6], v[7], v[8], v[9]]])

    center = np.linalg.solve(- A[:3, :3], v[6:9])

    translation_matrix = np.eye(4)
    translation_matrix[3, :3] = center.T

    R = translation_matrix.dot(A).dot(translation_matrix.T)

    evals, evecs = np.linalg.eig(R[:3, :3] / -R[3, 3])
    evecs = evecs.T

    radii = np.sqrt(1. / np.abs(evals))
    #radii *= np.sign(evals)

    return center, evecs, radii, v

In [ ]:
def set_axes_equal(ax: plt.Axes):
    ax.set_box_aspect([1,1,1])
    limits = np.array([
        ax.get_xlim3d(),
        ax.get_ylim3d(),
        ax.get_zlim3d(),
    ])
    x, y, z = np.mean(limits, axis=1)
    radius = 0.5 * np.max(np.abs(limits[:, 1] - limits[:, 0]))
    ax.set_xlim3d([x - radius, x + radius])
    ax.set_ylim3d([y - radius, y + radius])
    ax.set_zlim3d([z - radius, z + radius])

---

In [ ]:
layout = go.Layout(margin={'l': 0, 'r': 0, 'b': 0, 't': 0})

In [ ]:
def plot_3d_interactive_ellipse2(xyz_x, xyz_y, xyz_z, xx=[], yy=[], zz=[], dd=0.1):

    main_trace1 = go.Scatter3d(x=xyz_x, y=xyz_y, z=xyz_z,
                               mode='markers', marker=dict(size=5, color='blue', opacity=0.01),
                               name='Ellipsoid fit')

    # Create a shadow effect by adding a second scatter plot with offset coordinates
    shadow_trace1 = go.Scatter3d(x=[x + dd for x in xyz_x], y=[y + dd for y in xyz_y], z=[z - dd for z in xyz_z],
                                 mode='markers', marker=dict(size=5, color='red', opacity=0.5),
                                 name='Ellipsoid fit (shadows)')
    
    
    main_trace2 = go.Scatter3d(x=xx, y=yy, z=zz,
                               mode='markers', marker=dict(size=5, color='yellow', opacity=0.01),
                               name='Patch')

    # Create a shadow effect by adding a second scatter plot with offset coordinates
    shadow_trace2 = go.Scatter3d(x=[x + dd for x in xx], y=[y + dd for y in yy], z=[z - dd for z in zz],
                                 mode='markers', marker=dict(size=5, color='orange', opacity=0.3),
                                 name='Patch (shadows)')

    # Define the layout
    layout = go.Layout(
        scene=dict(xaxis=dict(title='X-axis'),
                   yaxis=dict(title='Y-axis'),
                   zaxis=dict(title='Z-axis')),
        title="Interactive 3D Scatter Plot with Shadows")

    # Create the figure
    fig = go.Figure(data=[shadow_trace1, main_trace1, shadow_trace2, main_trace2], layout=layout)

    # Display the plot
    pyo.plot(fig)

---

In [ ]:
# https://github.com/minillinim/ellipsoid
def ellipsoid_plot_grid(center, radii, rotation, u, v):
    
    # cartesian coordinates that correspond to the spherical angles:
    x = radii[0] * np.outer(np.cos(u), np.sin(v))
    y = radii[1] * np.outer(np.sin(u), np.sin(v))
    z = radii[2] * np.outer(np.ones_like(u), np.cos(v))
    # rotate accordingly
    lx = len(x)
    for i in range(lx):
        for j in range(lx):
            [x[i, j], y[i, j], z[i, j]] = np.dot([x[i, j], y[i, j], z[i, j]], rotation) + center
    
    return x.flatten().tolist(), y.flatten().tolist(), z.flatten().tolist()

---
---
---